# F1 Pit-Stop Prediction — Vanilla ML Reproduction

This notebook reproduces the requested **non-AutoML** approach using only the supplied competition files:

- `train(3).csv`
- `test(3).csv`
- `sample_submission(3).csv`

**Important constraints followed:**

- No `f1_strategy_dataset_v4.csv`
- No AutoGluon / AutoML
- No manual feature engineering
- No target encoding
- No arithmetic/interaction/lag features
- `Driver` is dropped from both train and test
- `eval_metric` is ROC-AUC
- The feature set is exactly the requested 13 columns
- Uses vanilla implementations of the model families named in the original solution: LightGBM, XGBoost, CatBoost, Random Forest and Extra Trees
- Uses a validation-based weighted ensemble, analogous to the single weighted-ensemble layer (`num_stack_levels=0`) in the requested AutoGluon setup

> **Performance note:** AutoGluon's `best_quality` automatically searches/tunes many configurations and creates its own optimized ensemble. A vanilla implementation cannot honestly guarantee the exact `0.959526` ROC-AUC without reproducing those hidden/tuned configurations. This notebook is designed to give a strong, transparent vanilla baseline while keeping the feature/data restrictions intact.


## 1. Install dependencies

Run this cell once in your local environment if required.

In [ ]:
# Uncomment if these packages are not installed
# %pip install pandas numpy scikit-learn lightgbm xgboost catboost matplotlib seaborn

## 2. Imports and configuration

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42
VALID_SIZE = 0.20

# Requested target / feature set
TARGET = "PitNextLap"

FEATURES = [
    "Compound",
    "Race",
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
]

CATEGORICAL_FEATURES = ["Compound", "Race"]
NUMERICAL_FEATURES = [c for c in FEATURES if c not in CATEGORICAL_FEATURES]

print("Features:", FEATURES)


Features: ['Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']


## 3. Load the supplied competition files

The notebook deliberately does **not** load or merge the external `f1_strategy_dataset_v4.csv` dataset.

In [2]:
TRAIN_PATH = r"D:\Work\DSA_Coding_Questions\F1_Pit_prediction\data\train.csv"
TEST_PATH = r"D:\Work\DSA_Coding_Questions\F1_Pit_prediction\data\test.csv"
SUBMISSION_PATH = r"D:\Work\DSA_Coding_Questions\F1_Pit_prediction\data\sample_submission.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SUBMISSION_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)


Train shape: (439140, 16)
Test shape: (188165, 15)
Sample submission shape: (188165, 2)


## 4. Basic cleanup — exactly as requested

In [3]:
# Drop ID
train = train.drop(columns=["id"])
test = test.drop(columns=["id"])

# Drop Driver because of train/test category mismatch
train = train.drop(columns=["Driver"])
test = test.drop(columns=["Driver"])

print("Train shape after cleanup:", train.shape)
print("Test shape after cleanup:", test.shape)

# Keep exactly the requested feature columns + target
X = train[FEATURES].copy()
y = train[TARGET].copy()
X_test = test[FEATURES].copy()

print("\nFeature columns:")
print(FEATURES)
print("\nTarget distribution:")
print(y.value_counts(normalize=True).sort_index())


Train shape after cleanup: (439140, 14)
Test shape after cleanup: (188165, 13)

Feature columns:
['Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']

Target distribution:
PitNextLap
0.0    0.801018
1.0    0.198982
Name: proportion, dtype: float64


## 5. Sanity checks

These checks do not modify the data.

In [4]:
print("Missing values:")
display(train[FEATURES + [TARGET]].isnull().sum().to_frame("missing"))

print("\nExact duplicate rows:", train.duplicated().sum())

print("\nTrain/test categorical differences:")
for col in CATEGORICAL_FEATURES:
    train_values = set(train[col].dropna().unique())
    test_values = set(test[col].dropna().unique())
    print(
        f"{col}: "
        f"train-only={len(train_values - test_values)}, "
        f"test-only={len(test_values - train_values)}"
    )


Missing values:


,missing
Compound,0
Race,0
Year,0
PitStop,0
LapNumber,0
Stint,0
TyreLife,0
Position,0
LapTime (s),0
LapTime_Delta,0



Exact duplicate rows: 14

Train/test categorical differences:
Compound: train-only=0, test-only=0
Race: train-only=0, test-only=0


## 6. Train/validation split

A fixed stratified split is used so that every vanilla model is evaluated on exactly the same validation rows.

In [5]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=VALID_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)


X_train: (351312, 13)
X_valid: (87828, 13)


## 7. Categorical handling

No manual feature engineering is performed.

For **LightGBM** and **CatBoost**, categorical columns are supplied directly.

For **XGBoost / Random Forest / Extra Trees**, the two categorical columns are converted with an ordinal encoder. Unknown categories in the test/validation data are encoded safely as `-1`.

Missing values are retained/handled by the model or preprocessing pipeline rather than being manually dropped.


In [6]:
# Shared ordinal encoder for sklearn/XGBoost models
ordinal_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OrdinalEncoder(
                    handle_unknown="use_encoded_value",
                    unknown_value=-1
                )),
            ]),
            CATEGORICAL_FEATURES,
        ),
        (
            "num",
            SimpleImputer(strategy="median"),
            NUMERICAL_FEATURES,
        ),
    ],
    remainder="drop",
)

X_train_encoded = ordinal_preprocessor.fit_transform(X_train)
X_valid_encoded = ordinal_preprocessor.transform(X_valid)
X_test_encoded = ordinal_preprocessor.transform(X_test)

encoded_feature_names = CATEGORICAL_FEATURES + NUMERICAL_FEATURES
print("Encoded matrix:", X_train_encoded.shape)


Encoded matrix: (351312, 13)


## 8. Model 1 — LightGBM

A strong vanilla gradient-boosting model. The parameters are explicit rather than being discovered by AutoML.

In [7]:
lgbm = LGBMClassifier(
    objective="binary",
    n_estimators=5000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

X_train_lgb = X_train.copy()
X_valid_lgb = X_valid.copy()

for c in CATEGORICAL_FEATURES:
    X_train_lgb[c] = X_train_lgb[c].astype("category")
    X_valid_lgb[c] = X_valid_lgb[c].astype("category")

lgbm.fit(
    X_train_lgb,
    y_train,
    categorical_feature=CATEGORICAL_FEATURES,
    eval_set=[(X_valid_lgb, y_valid)],
    callbacks=[
        early_stopping(150, verbose=False),
        log_evaluation(0),
    ],
)

pred_lgbm = lgbm.predict_proba(X_valid_lgb)[:, 1]
auc_lgbm = roc_auc_score(y_valid, pred_lgbm)

print("LightGBM best iteration:", lgbm.best_iteration_)
print(f"LightGBM validation ROC-AUC: {auc_lgbm:.6f}")


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 69905, number of negative: 281407
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005779 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1281
[LightGBM] [Info] Number of data points in the train set: 351312, number of used features: 13
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198983 -> initscore=-1.392665
[LightGBM] [Info] Start training from score -1.392665
LightGBM best iteration: 4128
LightGBM validation ROC-AUC: 0.950108


## 9. Model 2 — XGBoost

In [8]:
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    n_estimators=5000,
    learning_rate=0.03,
    max_depth=7,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

xgb.fit(
    X_train_encoded,
    y_train,
    eval_set=[(X_valid_encoded, y_valid)],
    verbose=False,
)

pred_xgb = xgb.predict_proba(X_valid_encoded)[:, 1]
auc_xgb = roc_auc_score(y_valid, pred_xgb)

print(f"XGBoost validation ROC-AUC: {auc_xgb:.6f}")


XGBoost validation ROC-AUC: 0.948118


## 10. Model 3 — CatBoost

In [9]:
# CatBoost handles the categorical columns directly.
X_train_cb = X_train.copy()
X_valid_cb = X_valid.copy()

# CatBoost requires categorical values to be strings and does not accept NaN
# inside categorical columns, so only categorical missing values are represented
# as a literal category. Numeric missing values are left for CatBoost.
for c in CATEGORICAL_FEATURES:
    X_train_cb[c] = X_train_cb[c].fillna("__MISSING__").astype(str)
    X_valid_cb[c] = X_valid_cb[c].fillna("__MISSING__").astype(str)

catboost = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=5000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3.0,
    random_seed=RANDOM_STATE,
    verbose=False,
    thread_count=-1,
    allow_writing_files=False,
)

catboost.fit(
    X_train_cb,
    y_train,
    cat_features=CATEGORICAL_FEATURES,
    eval_set=(X_valid_cb, y_valid),
    use_best_model=True,
    early_stopping_rounds=150,
    verbose=False,
)

pred_cat = catboost.predict_proba(X_valid_cb)[:, 1]
auc_cat = roc_auc_score(y_valid, pred_cat)

print(f"CatBoost validation ROC-AUC: {auc_cat:.6f}")


CatBoost validation ROC-AUC: 0.949056


## 11. Model 4 — Random Forest

In [10]:
rf = RandomForestClassifier(
    n_estimators=700,
    criterion="entropy",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    class_weight=None,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf.fit(X_train_encoded, y_train)

pred_rf = rf.predict_proba(X_valid_encoded)[:, 1]
auc_rf = roc_auc_score(y_valid, pred_rf)

print(f"Random Forest validation ROC-AUC: {auc_rf:.6f}")


Random Forest validation ROC-AUC: 0.945241


## 12. Model 5 — Extra Trees

In [11]:
extra = ExtraTreesClassifier(
    n_estimators=700,
    criterion="entropy",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=False,
    class_weight=None,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

extra.fit(X_train_encoded, y_train)

pred_extra = extra.predict_proba(X_valid_encoded)[:, 1]
auc_extra = roc_auc_score(y_valid, pred_extra)

print(f"Extra Trees validation ROC-AUC: {auc_extra:.6f}")


Extra Trees validation ROC-AUC: 0.942487


## 13. Compare individual models

In [12]:
model_results = pd.DataFrame({
    "Model": [
        "LightGBM",
        "XGBoost",
        "CatBoost",
        "RandomForest",
        "ExtraTrees",
    ],
    "Validation_ROC_AUC": [
        auc_lgbm,
        auc_xgb,
        auc_cat,
        auc_rf,
        auc_extra,
    ],
}).sort_values("Validation_ROC_AUC", ascending=False)

display(model_results)


,Model,Validation_ROC_AUC
0,LightGBM,0.950108
2,CatBoost,0.949056
1,XGBoost,0.948118
3,RandomForest,0.945241
4,ExtraTrees,0.942487


## 14. Weighted vanilla ensemble

This is the transparent replacement for AutoGluon's weighted ensemble layer.

The weights below are fixed and normalized. They are **not** produced by AutoML.

The ensemble combines the same model families used above and evaluates the combined probability on the untouched validation set.


In [13]:
# Fixed, transparent blend.
# Higher weight is given to the stronger boosting models.
weights = {
    "LightGBM": 0.30,
    "XGBoost": 0.25,
    "CatBoost": 0.20,
    "RandomForest": 0.15,
    "ExtraTrees": 0.10,
}

weight_sum = sum(weights.values())
weights = {k: v / weight_sum for k, v in weights.items()}

pred_ensemble = (
    weights["LightGBM"] * pred_lgbm
    + weights["XGBoost"] * pred_xgb
    + weights["CatBoost"] * pred_cat
    + weights["RandomForest"] * pred_rf
    + weights["ExtraTrees"] * pred_extra
)

auc_ensemble = roc_auc_score(y_valid, pred_ensemble)

print("Ensemble weights:")
for name, weight in weights.items():
    print(f"  {name}: {weight:.3f}")

print(f"\nWeighted ensemble validation ROC-AUC: {auc_ensemble:.6f}")


Ensemble weights:
  LightGBM: 0.300
  XGBoost: 0.250
  CatBoost: 0.200
  RandomForest: 0.150
  ExtraTrees: 0.100

Weighted ensemble validation ROC-AUC: 0.950656


## 15. Train final models on all available competition training data

Once the validation experiment is complete, the final models are retrained on the complete original competition `train.csv`.

No external dataset is added.

Early stopping is intentionally not used in the final fit because there is no final validation set. The number of boosting iterations learned during the validation stage is reused.


In [14]:
# Prepare complete training data
X_full = train[FEATURES].copy()
y_full = train[TARGET].copy()
X_test_full = test[FEATURES].copy()

# ----- LightGBM -----
X_full_lgb = X_full.copy()
X_test_lgb = X_test_full.copy()

for c in CATEGORICAL_FEATURES:
    # Use a common category space so train/test categories remain compatible.
    combined = pd.concat([X_full_lgb[c], X_test_lgb[c]], axis=0).astype("category")
    categories = combined.cat.categories
    X_full_lgb[c] = pd.Categorical(X_full_lgb[c], categories=categories)
    X_test_lgb[c] = pd.Categorical(X_test_lgb[c], categories=categories)

lgbm_final = LGBMClassifier(
    objective="binary",
    n_estimators=int(lgbm.best_iteration_ or 1000),
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

lgbm_final.fit(
    X_full_lgb,
    y_full,
    categorical_feature=CATEGORICAL_FEATURES,
    callbacks=[log_evaluation(0)],
)

final_lgbm_test = lgbm_final.predict_proba(X_test_lgb)[:, 1]

# ----- sklearn preprocessing for XGBoost / RF / ExtraTrees -----
final_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OrdinalEncoder(
                    handle_unknown="use_encoded_value",
                    unknown_value=-1
                )),
            ]),
            CATEGORICAL_FEATURES,
        ),
        (
            "num",
            SimpleImputer(strategy="median"),
            NUMERICAL_FEATURES,
        ),
    ],
    remainder="drop",
)

X_full_encoded = final_preprocessor.fit_transform(X_full)
X_test_encoded = final_preprocessor.transform(X_test_full)

# ----- XGBoost -----
xgb_final = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    n_estimators=int(xgb.get_booster().num_boosted_rounds()),
    learning_rate=0.03,
    max_depth=7,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

xgb_final.fit(X_full_encoded, y_full, verbose=False)
final_xgb_test = xgb_final.predict_proba(X_test_encoded)[:, 1]

# ----- CatBoost -----
X_full_cb = X_full.copy()
X_test_cb = X_test_full.copy()

for c in CATEGORICAL_FEATURES:
    X_full_cb[c] = X_full_cb[c].fillna("__MISSING__").astype(str)
    X_test_cb[c] = X_test_cb[c].fillna("__MISSING__").astype(str)

cat_final = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=int(catboost.get_best_iteration() + 1),
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3.0,
    random_seed=RANDOM_STATE,
    verbose=False,
    thread_count=-1,
    allow_writing_files=False,
)

cat_final.fit(
    X_full_cb,
    y_full,
    cat_features=CATEGORICAL_FEATURES,
    verbose=False,
)

final_cat_test = cat_final.predict_proba(X_test_cb)[:, 1]

# ----- Random Forest -----
rf_final = RandomForestClassifier(
    n_estimators=700,
    criterion="entropy",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    class_weight=None,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_final.fit(X_full_encoded, y_full)
final_rf_test = rf_final.predict_proba(X_test_encoded)[:, 1]

# ----- Extra Trees -----
extra_final = ExtraTreesClassifier(
    n_estimators=700,
    criterion="entropy",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=False,
    class_weight=None,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

extra_final.fit(X_full_encoded, y_full)
final_extra_test = extra_final.predict_proba(X_test_encoded)[:, 1]

print("Final models trained on:", X_full.shape)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 87381, number of negative: 351759
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013425 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1282
[LightGBM] [Info] Number of data points in the train set: 439140, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198982 -> initscore=-1.392668
[LightGBM] [Info] Start training from score -1.392668
Final models trained on: (439140, 13)


## 16. Create final submission

In [15]:
final_test_prediction = (
    weights["LightGBM"] * final_lgbm_test
    + weights["XGBoost"] * final_xgb_test
    + weights["CatBoost"] * final_cat_test
    + weights["RandomForest"] * final_rf_test
    + weights["ExtraTrees"] * final_extra_test
)

submission = sample_submission.copy()
submission[TARGET] = final_test_prediction

OUTPUT_PATH = "best_vanilla_quality.csv"
submission.to_csv(OUTPUT_PATH, index=False)

print(f"Saved submission: {OUTPUT_PATH}")
display(submission.head())
print("\nSubmission shape:", submission.shape)
print("Prediction range:", submission[TARGET].min(), "to", submission[TARGET].max())


Saved submission: best_vanilla_quality.csv


,id,PitNextLap
0,439140,0.006529
1,439141,0.002824
2,439142,0.002773
3,439143,0.203460
4,439144,0.902585



Submission shape: (188165, 2)
Prediction range: 2.9755297883188828e-05 to 0.9947521528321798


## 17. Final checks

The output is a probability submission for class `1` (`PitNextLap`), matching the requested ROC-AUC competition format.

Expected columns:

- submission ID column from `sample_submission.csv`
- `PitNextLap` probability


In [16]:
assert len(submission) == len(test)
assert TARGET in submission.columns
assert submission[TARGET].between(0, 1).all()

print("PASS: submission row count matches test.csv")
print("PASS: PitNextLap contains probabilities in [0, 1]")
print("PASS: submission saved as:", OUTPUT_PATH)


PASS: submission row count matches test.csv
PASS: PitNextLap contains probabilities in [0, 1]
PASS: submission saved as: best_vanilla_quality.csv
